# Agent with Long-Term Memory (LangGraph)

MBA project - Agents, Multi-Agents and Interoperability
Author: Yuri Telles Ferreira

This agent remembers the customer between different conversations. When the customer tells
something (a preference or an order), the agent saves it in a Store. Later, in a new thread,
the agent has no chat history. But it can search the Store and give the right answer.
I tested it with two customers. One customer cannot see the data of the other customer.

## Install the libraries

In [ ]:
!pip install -U langgraph langgraph-checkpoint-sqlite langchain-openai -q


## Imports and API key

The OpenAI key comes from the Colab secrets.

In [ ]:
import os
import uuid
import sqlite3
from typing import Annotated

from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition, InjectedStore
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.sqlite import SqliteStore
from langgraph.store.base import BaseStore

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


## Two types of memory

The checkpointer saves the chat history for each thread. The Store is the important part here:
it saves data for each customer, and the data stays when the thread changes. I use SQLite, so
the data is saved in a file.

In [ ]:
conn = sqlite3.connect("customer_memory.db", check_same_thread=False, autocommit=True)

store = SqliteStore(
    conn,
    index={"embed": "openai:text-embedding-3-small", "dims": 1536},
)
store.setup()

checkpointer = MemorySaver()

print("Store and checkpointer are ready.")


## Tools: save and search

There are two tools. One tool saves what the customer says. The other tool searches by semantic
similarity (it does not need the exact words). The user_id comes from the config of the call.
The model does not choose the user_id.

In [ ]:
@tool
def save_memory(
    memory: str,
    config: RunnableConfig,
    *,
    store: Annotated[BaseStore, InjectedStore()],
) -> str:
    """Save a preference, personal data or an order of the current customer in long-term memory.
    Use this tool every time the customer tells something important about themselves
    (preferences, personal data) or makes an order."""
    user_id = config["configurable"]["user_id"]
    namespace = ("memories", user_id)
    key = str(uuid.uuid4())
    store.put(namespace, key, {"data": memory})
    return f"Memory saved for customer '{user_id}': {memory}"


@tool
def search_memory(
    query: str,
    config: RunnableConfig,
    *,
    store: Annotated[BaseStore, InjectedStore()],
) -> str:
    """Search, by semantic similarity, the saved memories about the current customer
    (who they are, preferences, past orders). ALWAYS use this tool before you answer
    questions like 'who am I?' or 'what is my order?'."""
    user_id = config["configurable"]["user_id"]
    namespace = ("memories", user_id)
    results = store.search(namespace, query=query, limit=5)
    if not results:
        return f"No memories found for customer '{user_id}'."
    return "\n".join(f"- {item.value['data']}" for item in results)


tools = [save_memory, search_memory]


## Build the graph

A simple ReAct flow: the agent decides to call a tool or to answer directly. The tool runs and
then goes back to the agent.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = """You are the virtual assistant of the store "Quantum Commerce".
You are helping the customer with the ID "{user_id}".

Required rules:
1. When the customer tells a preference or personal data, or makes an order,
   call the tool `save_memory` to save this information.
2. When the customer asks who they are, what their preferences are or what their
   order was, FIRST call the tool `search_memory`, and then answer.
   Never invent information: use only what the search returns.
3. Be short, polite and direct in your answers.
"""


def call_agent(state: MessagesState, config: RunnableConfig):
    user_id = config["configurable"]["user_id"]
    system_message = SystemMessage(content=SYSTEM_PROMPT.format(user_id=user_id))
    response = llm_with_tools.invoke([system_message] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("agent", call_agent)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    tools_condition,
    {"tools": "tools", END: END},
)
builder.add_edge("tools", "agent")

graph = builder.compile(checkpointer=checkpointer, store=store)
print("Graph compiled.")


## Function to chat with the agent

In [ ]:
def chat(message: str, thread_id: str, user_id: str):
    config = {"configurable": {"thread_id": thread_id, "user_id": user_id}}
    result = graph.invoke({"messages": [("user", message)]}, config=config)
    response = result["messages"][-1].content
    print(f"[{user_id} | {thread_id}] Customer: {message}")
    print(f"[{user_id} | {thread_id}] Agent:    {response}\n")
    return response


## Customer 1 - Ana

First chat with Ana, in one thread. Here we only save her data.

In [ ]:
chat(
    "Hi! My name is Ana Clara, I live in Sao Paulo and I really like coffee without sugar.",
    thread_id="ana-thread-1",
    user_id="ana",
)

chat(
    "I want to make an order: 2 boxes of dark chocolate and 1 pack of decaf coffee.",
    thread_id="ana-thread-1",
    user_id="ana",
)


## The real test: a new thread for Ana

This is the key part: a new thread, with no chat history. If the agent gives the right answer,
it is because it searched the Store, not because it remembers the chat.

In [ ]:
chat(
    "Who am I and what was my order?",
    thread_id="ana-thread-2-NEW",
    user_id="ana",
)


## Customer 2 - Bruno

The same steps, with another customer.

In [ ]:
chat(
    "Hello, I am Bruno Silva, I live in Curitiba and I love vegan products.",
    thread_id="bruno-thread-1",
    user_id="bruno",
)

chat(
    "My order is: 3 vegan chocolate bars and 1 box of green tea.",
    thread_id="bruno-thread-1",
    user_id="bruno",
)


## The real test: a new thread for Bruno

In [ ]:
chat(
    "Who am I and what was my order?",
    thread_id="bruno-thread-2-NEW",
    user_id="bruno",
)


## Check the Store directly

This shows that the data of the two customers does not mix.

In [ ]:
for customer in ["ana", "bruno"]:
    print(f"=== Saved memories for '{customer}' ===")
    results = store.search(("memories", customer), query="customer data", limit=10)
    for item in results:
        print(f"Key: {item.key} -> {item.value}")
    print()


## Conclusion

The project does what the task asked:
- the agent knows who the customer is and what their order was;
- it saves this information in the Store;
- it searches by meaning (semantic search), not by an exact key;
- it works in a new thread with no chat history;
- I tested it with 2 different customers, and their data does not mix.